# Dataset statistics

Count and group-by queries with matplotlib charts.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from causal_ai_av.dataset import CausalAVDataset

ds = CausalAVDataset(Path("/home/horde/01_json_annotations"))
print(f"corpus has {len(ds)} clips")


## Headline counts


In [ ]:
rows = [
    ("pedestrian", ds.count("agent.type = ped")),
    ("vehicle", ds.count("agent.type = vehicle")),
    ("VRU", ds.count("agent.type = vru")),
    ("cyclist", ds.count("agent.type = cyclist")),
    ("stop sign", ds.count("obj.type = stop_sign")),
    ("traffic light (red)", ds.count("light.color = red")),
    ("crosswalk env", ds.count("env.type = crosswalk")),
    ("intersection env", ds.count("env.type = intersection")),
    ("roundabout env", ds.count("env.type = roundabout")),
    ("multi-lane road", ds.count("env(type = road, lanes >= 2)")),
    ("ego decel", ds.count("ego.action = decel")),
    ("ego stops at red", ds.count("light.color = red and ego.action = stop")),
]
pd.DataFrame(rows, columns=["query", "clips"]).sort_values("clips", ascending=False)


## Distribution of agent types


In [ ]:
agent_dist = ds.group_by(
    "agent.type = vehicle or agent.type = vru or agent.type = animal",
    key="agent.type",
)
s = pd.Series(agent_dist).sort_values(ascending=True)
ax = s.plot(kind="barh", figsize=(8, 6))
ax.set_xlabel("clips")
ax.set_title("Clips per agent type")
plt.tight_layout()
plt.show()


## Distribution of ego actions


In [ ]:
ego_dist = ds.group_by(
    "ego.action.type = drive or ego.action.type = stop or ego.action.type = decel "
    "or ego.action.type = yield or ego.action.type = turn_left "
    "or ego.action.type = turn_right or ego.action.type = change_lane",
    key="ego.action.type",
)
s = pd.Series(ego_dist).sort_values(ascending=True)
ax = s.plot(kind="barh", figsize=(8, 6))
ax.set_xlabel("clips")
ax.set_title("Clips per ego-action type")
plt.tight_layout()
plt.show()


## Distribution of environment types


In [ ]:
env_dist = ds.group_by(
    "env.type = road or env.type = intersection or env.type = crosswalk "
    "or env.type = sidewalk or env.type = roundabout or env.type = cycle_lane "
    "or env.type = tunnel",
    key="env.type",
)
s = pd.Series(env_dist).sort_values(ascending=True)
ax = s.plot(kind="barh", figsize=(8, 5))
ax.set_xlabel("clips")
ax.set_title("Clips per environment type")
plt.tight_layout()
plt.show()


## Composite questions


In [ ]:
n_red = ds.count("light.color = red")
n_red_stop = ds.count("light.color = red and ego.action = stop")
n_yellow_clear = ds.count("light(color = yellow, could_have_cleared = true)")
n_jaywalk = ds.count("agent(type = ped, action(jaywalk = true))")

composite = pd.DataFrame([
    ("ego stops at red", f"{n_red_stop}/{n_red}", f"{100*n_red_stop/max(n_red,1):.0f}%"),
    ("yellow ego could have cleared", n_yellow_clear, ""),
    ("jaywalking pedestrian", n_jaywalk, ""),
], columns=["question", "count", "ratio"])
composite
